In [1]:
# Import Spark Session
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("Week6_Assignment") \
    .getOrCreate()

print("Spark Session Created")

Spark Session Created


In [2]:
# Sample dataset
data = [
    (101,"Electronics","1500"),
    (102,"Books","500"),
    (103,"Electronics","3000"),
    (104,"Clothes","700")
]

columns = ["product_id","category","price"]

# Create DataFrame
df = spark.createDataFrame(data,columns)

# Display DataFrame
df.show()

+----------+-----------+-----+
|product_id|   category|price|
+----------+-----------+-----+
|       101|Electronics| 1500|
|       102|      Books|  500|
|       103|Electronics| 3000|
|       104|    Clothes|  700|
+----------+-----------+-----+



In [3]:
import pandas as pd

# Create sample csv file
sample = pd.DataFrame({
    "product_id":[101,102,103,104],
    "category":["Electronics","Books","Electronics","Clothes"],
    "price":[1500,500,3000,700]
})

sample.to_csv("source.csv",index=False)

# Read CSV using Spark
df_csv = spark.read \
    .option("header",True) \
    .option("inferSchema",True) \
    .csv("source.csv")

df_csv.show()

+----------+-----------+-----+
|product_id|   category|price|
+----------+-----------+-----+
|       101|Electronics| 1500|
|       102|      Books|  500|
|       103|Electronics| 3000|
|       104|    Clothes|  700|
+----------+-----------+-----+



In [4]:
electronics = df.select(
    "product_id",
    "price"
).filter(
    df.category=="Electronics"
)

electronics.show()

+----------+-----+
|product_id|price|
+----------+-----+
|       101| 1500|
|       103| 3000|
+----------+-----+



In [5]:
from pyspark.sql.functions import col

# Rename column
df = df.withColumnRenamed(
    "product_id",
    "new_name"
)

# Cast datatype
df = df.withColumn(
    "price",
    col("price").cast("double")
)

df.printSchema()

root
 |-- new_name: long (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)



In [6]:
df = df.withColumn(
    "final_price",
    col("price")*1.18
)

df.show()

+--------+-----------+------+-----------+
|new_name|   category| price|final_price|
+--------+-----------+------+-----------+
|     101|Electronics|1500.0|     1770.0|
|     102|      Books| 500.0|      590.0|
|     103|Electronics|3000.0|     3540.0|
|     104|    Clothes| 700.0|      826.0|
+--------+-----------+------+-----------+



In [7]:
orders = [
    (1,"Completed",2000),
    (2,"Pending",500),
    (3,"Completed",1500),
    (4,"Completed",400)
]

df_orders = spark.createDataFrame(
    orders,
    ["id","status","amount"]
)

result = df_orders.filter(
    (df_orders.status=="Completed") &
    (df_orders.amount>1000)
)

result.show()

+---+---------+------+
| id|   status|amount|
+---+---------+------+
|  1|Completed|  2000|
|  3|Completed|  1500|
+---+---------+------+



In [8]:
data = [
    (1,"A"),
    (2,"B"),
    (None,"C")
]

df = spark.createDataFrame(
    data,
    ["user_id","name"]
)

# Save parquet
df.write.mode("overwrite").parquet("input_parquet")

# Read parquet
df2 = spark.read.parquet(
    "input_parquet"
)

# Filter null values
df2.filter(
    df2.user_id.isNotNull()
).write.mode(
    "overwrite"
).csv(
    "output_csv"
)

print("Saved Successfully")

+-------+----+
|user_id|name|
+-------+----+
|      1|   A|
|      2|   B|
|   NULL|   C|
|      4|   D|
+-------+----+

+-------+----+
|user_id|name|
+-------+----+
|      1|   A|
|      2|   B|
|      4|   D|
+-------+----+



In [9]:
data = [
    ("North","Low"),
    ("South","High"),
    ("North","High"),
    ("East","Low")
]

df_region = spark.createDataFrame(
    data,
    ["region","priority"]
)

result = df_region.filter(
    (df_region.region=="North") |
    (df_region.priority=="High")
)

result.show()

+------+--------+
|region|priority|
+------+--------+
| North|     Low|
| South|    High|
| North|    High|
+------+--------+



In [10]:
# Save as parquet
df.write.mode("overwrite").parquet(
    "output_parquet"
)

# Read parquet
parquet_df = spark.read.parquet(
    "output_parquet"
)

parquet_df.show()

+--------+-----------+------+-----------+
|new_name|   category| price|final_price|
+--------+-----------+------+-----------+
|     103|Electronics|3000.0|     3540.0|
|     104|    Clothes| 700.0|      826.0|
|     101|Electronics|1500.0|     1770.0|
|     102|      Books| 500.0|      590.0|
+--------+-----------+------+-----------+



In [11]:
pipeline = spark.read \
    .option("header",True) \
    .csv("source.csv")

pipeline = pipeline.withColumn(
    "price",
    col("price").cast("double")
)

pipeline = pipeline.filter(
    col("price") > 1000
)

pipeline.write.mode(
    "overwrite"
).parquet(
    "final_output"
)

print("Pipeline Executed Successfully")

Pipeline Executed Successfully
